In [ ]:
%matplotlib inline
import os
import cv2
from skimage.morphology import remove_small_objects
import scipy.ndimage as ndimage
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
import numpy as np
from tifffile import tifffile
import ipywidgets as widgets
from ipywidgets import interact

In [ ]:
IMAGE_PATH = f"/net/birdstore/Active_Atlas_Data/data_root/pipeline_data/DK283/preps/C2/cropped"

In [ ]:
def process_and_display(infile, gain=1, kernel_size=0, threshold=0):
    image_inpath = os.path.join(IMAGE_PATH, infile)
    if not os.path.exists(image_inpath):
        print(f'Missing image path {image_inpath}')
        return
    with tifffile.TiffFile(image_inpath) as tif:
        img = tif.asarray()

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    #left side image, original * gain so you can see it
    modified_img = img.astype(np.uint32) * gain
    gained_img = np.clip(modified_img, 0, 65535).astype(np.uint16)

    _max = img.max()
    axes[0].imshow(gained_img, cmap="gray")
    axes[0].set_title(f"Original {infile} with gain: {gain}, max before gain: {_max}")
    axes[0].axis("off")
    
    binary_mask = img > threshold
    # Apply morphological opening
    if kernel_size > 0:
        binary_mask = remove_small_objects(binary_mask, max_size=kernel_size)
    pixel_count = np.count_nonzero(binary_mask)
    int8_arr = (binary_mask * 255).astype(np.uint8)
    
    axes[1].imshow(int8_arr, cmap="gray")
    axes[1].set_title(f"Threshold: {threshold}, pixel count: {pixel_count}")
    axes[1].axis("off")

    min_val = int(np.min(img))
    max_val = int(np.max(img))
    step = 100
    thresholds = np.arange(min_val, max_val + step, step)
    pixel_sums = [np.sum(img[img > t].astype(np.int64)) for t in thresholds]
    axes[2].plot(thresholds, pixel_sums, color='darkviolet', linewidth=2)
    axes[2].set_title(f'Log Sum of Brightness Values vs. Threshold (Active Range: {min_val}:{max_val})')
    axes[2].set_xlabel('Threshold Value (Filtered to Active Range)', fontsize=12)
    axes[2].set_ylabel('Sum of Brightness Intensities', fontsize=12)
    axes[2].grid(True, linestyle='--', alpha=0.6)
    axes[2].set_yscale('log')
    #axes[2].set_xscale('log')
    axes[2].set_xlim(min_val, max_val)

    
    plt.tight_layout()
    plt.show()

In [ ]:
infile = "020.tif"
interact(process_and_display,
    infile = infile,
    gain=widgets.IntSlider(min=0, max=50, step=1, value=1, description="Gain"),
    kernel_size=widgets.IntSlider(min=0, max=1000, step=1, value=0, description="Min size"),
    threshold=widgets.IntSlider(min=0, max=65000, step=1, value=500, description="Threshold")
)


In [ ]:
infile = "020.tif"
image_inpath = os.path.join(IMAGE_PATH, infile)
with tifffile.TiffFile(image_inpath) as tif:
    img = tif.asarray()

# Filter structural 0s to avoid log(0) errors
positive_data = img[img > 0]
# Generate log-spaced bins 
num_bins = 150
bins = np.logspace(np.log10(positive_data.min()), np.log10(positive_data.max()), num=num_bins)
counts, edges = np.histogram(positive_data, bins=bins)
# Calculate centers of the bins
bin_centers = 10**((np.log10(edges[:-1]) + np.log10(edges[1:])) / 2)
# Transform into Log-Log Space ---
valid = counts > 0
log_x = np.log10(bin_centers[valid])
log_y = np.log10(counts[valid])
# Smooth & Compute Derivatives ---
window_length = 15  # Must be odd. Increase if data is highly jagged.
polyorder = 3
smoothed_log_y = savgol_filter(log_y, window_length, polyorder)
dy_dx = np.gradient(smoothed_log_y, log_x)
d2y_dx2 = np.gradient(dy_dx, log_x)
# Locate Max 2nd Derivative After Peak Noise ---
# Find the top of the noise distribution peak
noise_peak_idx = np.argmax(smoothed_log_y)
# Slice the 2nd derivative array to look ONLY after this peak
post_peak_d2y = d2y_dx2[noise_peak_idx:]
# Find the index where the 2nd derivative reaches its maximum value
max_curvature_relative_idx = np.argmax(post_peak_d2y)
inflection_idx = noise_peak_idx + max_curvature_relative_idx
# 4. Convert back from log-space to extract the exact intensity
axon_threshold = 10**log_x[inflection_idx]
print(f"🎯 Max Curvature Axon Threshold: {axon_threshold:.4f}")

# --- 5. Plotting ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 8), sharex=True)

# Top Plot: Log-Log Histogram & Threshold
ax1.bar(bin_centers, counts, width=np.diff(edges), alpha=0.4, color='gray', label='Raw Histogram')
ax1.plot(10**log_x, 10**smoothed_log_y, color='black', lw=2, label='Smoothed Trend')
ax1.axvline(axon_threshold, color='crimson', linestyle='--', lw=2, label=f'Threshold: {axon_threshold:.2f}')
ax1.set_yscale('log')
ax1.set_ylabel('Pixel Count (Log)', fontsize=11)
ax1.set_title(f'{infile } Axon Thresholding via Maximum 2nd Derivative', fontsize=13, fontweight='bold')
ax1.legend()

# Bottom Plot: Derivatives & Max Curvature Point
ax2.plot(10**log_x, dy_dx, color='darkorange', lw=1.5, label="x'")
ax2.plot(10**log_x, d2y_dx2, color='deepskyblue', lw=1.5, label="x''")
ax2.axhline(0, color='black', linestyle=':', alpha=0.6)

# Mark the maximum curvature inflection point
ax2.axvline(axon_threshold, color='crimson', linestyle='--', lw=2)
ax2.plot(axon_threshold, d2y_dx2[inflection_idx], 'ro', ms=8, label="Max x'' Point")

ax2.set_xscale('log')
ax2.set_xlabel('Intensity Value (Log)', fontsize=11)
ax2.set_ylabel('Derivative Magnitude', fontsize=11)
ax2.legend()

plt.tight_layout()
plt.show()
